In [62]:

import pandas as pd
import torch
from pandas import DataFrame

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# Data preparation

In [3]:
# Model parameters
HORIZON = 1
BATCH_SIZE = 800
NUM_EPOCHS = 25
HIDDEN_SIZE = 128
N_LAYERS = 3
DROPOUT = 0.3
EMBEDDING_SIZE = 32

# Train parameters
TARGET = "EXPORT_centered"
FEATURES = [
  "contig", "comlang_off", "colony", "smctry",  # dist cepii categorical
]
N_SPLITS = 5
PATIENCE = 5
LEARNING_RATE = 0.01
WEIGHT_DECAY = 0.01
RANDOM_SEED = 16
KEEP_FRAC = 1.0
N_LAGS = 5
SUBSAMPLE_ENABLED = False
N_DYADS = 1000

SANCTION_COLS = ["arms", "military", "trade", "travel", "other"]

# Torch config
torch.manual_seed(RANDOM_SEED)
device = (
  torch.device("mps") if torch.backends.mps.is_available()
  else torch.device("cpu")
)

In [4]:
processed = pd.read_parquet(path="../../data/model/processed.parquet", engine="fastparquet")

df: DataFrame = processed.copy(deep=True)
df["dyad_id"] = df["ISO3_reporter"] + "_" + df["ISO3_partner"]
df = df.sort_values(by=["dyad_id", "Year"], ignore_index=True)

# Prepare sanction column as sum of all active boolean sanctions
df["sanction"] = (df[SANCTION_COLS]
                  .sum(axis=1)).astype(int)

# Coerce numerical columns to float
num_cols = ["distw", "GDP_reporter", "GDP_partner", "sanction", "contig",
            "comlang_off", "colony", "smctry", "Year", ]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").astype(float)

# Drop NA in numerical columns
df = df.dropna(subset=num_cols)

# Cast "Year" to integer
df["Year"] = df["Year"].astype(int)

# Cast "dyad_id" to categorical
for col in ["dyad_id"]:
  df[col] = pd.Categorical(df[col], categories=sorted(df[col].unique()))

  # Define columns to be lagged and lag them while appending the number of the lag to the column name
lag_cols = ["GDP_reporter", "GDP_partner", "sanction"]
for col in lag_cols:
  for index in range(1, N_LAGS + 1):
    df[f"{col}_lag{index}"] = df.groupby("dyad_id", observed=True)[col].shift(index)

# Drop NA again for the lags that produced NA
df = df.dropna()

# Add lagged column names to the feature list
FEATURES += [f"{c}_lag{index}" for c in lag_cols for index in range(1, N_LAGS + 1)]

In [5]:
DYAD_PAIRS = [
  ("USA_CHN", "CHN_USA"),
  ("USA_CAN", "CAN_USA"),
  ("DEU_CHN", "CHN_DEU"),
  ("USA_DEU", "DEU_USA"),
  ("DEU_FRA", "FRA_DEU"),
]

# Check time series for stationarity and cointegration

In [74]:
# Define which dyad to investigate
dyad_id = "DEU_FRA"
dyad_df = df[df["dyad_id"] == dyad_id]

# Define which time series to investigate
ts_columns = [
  "GDP_reporter",
  "GDP_partner",
  "sanction",
  "EXPORT"
]

In [84]:
# Check for stationarity using Augmented Dickey-Fuller test
for col in ts_columns:

  try:
    result_adf = adfuller(dyad_df[col].values, autolag="AIC")
  except ValueError as e:
    print("!" * 50)
    print(f"⚠️ Column \"{col}\" constant over the whole dyad! Skipping!!")
    print(f"!" * 50 + "\n\n")
    continue

  adf_statistic = result_adf[0]
  p_value = result_adf[1]
  critical_values = result_adf[4]

  print(f"ADF Test for time series: {col}")
  print("=" * 50)

  print(f"p-value: {p_value}")
  print(f"ADF statistic: {adf_statistic}")
  print(
    f"Critical value 1%: {critical_values["1%"]}\nCritical value 5%: {critical_values["5%"]}\nCritical value 10%: {critical_values["10%"]}\n\n")

ADF Test for time series: GDP_reporter
p-value: 0.46677853089768784
ADF statistic: -1.6313607343858119
Critical value 1%: -3.7883858816542486
Critical value 5%: -3.013097747543462
Critical value 10%: -2.6463967573696143


ADF Test for time series: GDP_partner
p-value: 0.03498598751857147
ADF statistic: -2.9990203615146824
Critical value 1%: -3.7883858816542486
Critical value 5%: -3.013097747543462
Critical value 10%: -2.6463967573696143


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
⚠️ Column "sanction" constant over the whole dyad! Skipping!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


ADF Test for time series: EXPORT
p-value: 0.1434019543607543
ADF statistic: -2.3941232439307947
Critical value 1%: -3.7883858816542486
Critical value 5%: -3.013097747543462
Critical value 10%: -2.6463967573696143




In [85]:
# Check for stationarity using Kwiatkowski-Phillips-Schmidt-Shin test
for col in ts_columns:

  try:
    result_adf = kpss(dyad_df[col].values, regression="ct")
  except ValueError as error:
    print("!" * 50)
    print(f"⚠️ Column \"{col}\" constant over the whole dyad! Skipping!!")
    print(f"!" * 50 + "\n\n")
    continue

  kpss_statistic = result_adf[0]
  p_value = result_adf[1]
  critical_values = result_adf[3]

  print(f"KPSS Test for time series: {col}")
  print("=" * 50)

  print(f"p-value: {p_value}")
  print(f"KPSS statistic: {kpss_statistic}")
  print(
    f"Critical value 1%: {critical_values["1%"]}\nCritical value 2.5%: {critical_values["2.5%"]}\nCritical value 5%: {critical_values["5%"]}\nCritical value 10%: {critical_values["10%"]}\n\n")

KPSS Test for time series: GDP_reporter
p-value: 0.1
KPSS statistic: 0.08402665821387242
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119


KPSS Test for time series: GDP_partner
p-value: 0.07548385309564558
KPSS statistic: 0.13223871932835138
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
⚠️ Column "sanction" constant over the whole dyad! Skipping!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


KPSS Test for time series: EXPORT
p-value: 0.031228375209262756
KPSS statistic: 0.16852594974888468
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119




/var/folders/wz/kf7643gn3_s2867t_nnc9zxc0000gn/T/ipykernel_86636/2406316631.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result_adf = kpss(dyad_df[col].values, regression="ct")
/opt/homebrew/Caskroom/miniconda/base/envs/thesis_env/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:2182: RuntimeWarning: invalid value encountered in scalar divide
  s_hat = s1 / s0


In [93]:
''  # Check for cointegration using Johansen test
data_johansen = dyad_df[["GDP_partner", "GDP_reporter"]].dropna()

result = coint_johansen(data_johansen, det_order=0, k_ar_diff=5)
trace_statistic = result.lr1[0]
max_eigenvalue_statistic = result.lr2[0]
print(f"Johansen Test for cointegration between EXPORT and GDP_partner")
print("=" * 50)
print("Trace Statistics:", result.lr1)
print("Critical Values (Trace):", result.cvt)

Johansen Test for cointegration between EXPORT and GDP_partner
Trace Statistics: [51.96478376  1.22470334]
Critical Values (Trace): [[13.4294 15.4943 19.9349]
 [ 2.7055  3.8415  6.6349]]


# VECM (Vector Error Correction Models)

Because GDP and EXPORT are both non-stationary and cointegrated, we cannot run the normal Granger causality test. But, we can run VECM.